In [53]:
import pandas as pd

In [54]:
customers=pd.read_csv('Datos/Originales/Datos_look&like/customers_data_2.csv',sep=';')
items=pd.read_csv('Datos/Originales/Datos_look&like/items_data.csv')
lyl=pd.read_csv('Datos/Originales/Datos_look&like/look_and_like_data_2.csv',sep=';')


In [55]:
import pandas as pd
import numpy as np

# Función auxiliar para analizar un DataFrame individual
def check_quality(df, name):
    print(f"\n{'='*20} ANÁLISIS: {name} {'='*20}")
    print(f"Dimensiones: {df.shape}")
    
    # 1. Análisis de Nulos
    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df)) * 100
    nulos_df = pd.DataFrame({'Total Nulos': nulos, '% Nulos': nulos_pct})
    # Filtramos para mostrar solo columnas con problemas o todas si prefieres
    print("\n--- Valores Nulos ---")
    if nulos.sum() == 0:
        print("¡No hay valores nulos!")
    else:
        print(nulos_df[nulos_df['Total Nulos'] > 0])
    
    # 2. Tipos de datos (Para ver si los IDs son int, float o object)
    print("\n--- Tipos de Datos (Head) ---")
    print(df.dtypes)
    
    # 3. Duplicados
    duplicados = df.duplicated().sum()
    print(f"\n--- Filas totalmente duplicadas: {duplicados} ---")

# Ejecutamos el análisis individual
check_quality(customers, "CUSTOMERS")
check_quality(items, "ITEMS")
check_quality(lyl, "LOOK & LIKE")

print(f"\n{'='*20} VERIFICACIÓN DE CLAVES (CRUCIAL PARA MERGE) {'='*20}")

# 4. Comprobar consistencia de IDs (Integridad Referencial)
# ¿Cuántos usuarios de lyl NO están en customers?
missing_users = lyl[~lyl['user_id'].isin(customers['user_id'])]['user_id'].nunique()
total_users_lyl = lyl['user_id'].nunique()
print(f"Usuarios en 'lyl' que NO existen en 'customers': {missing_users} de {total_users_lyl}")

# ¿Cuántos productos de lyl NO están en items?
missing_items = lyl[~lyl['product_variant_id'].isin(items['product_variant_id'])]['product_variant_id'].nunique()
total_items_lyl = lyl['product_variant_id'].nunique()
print(f"Productos en 'lyl' que NO existen en 'items': {missing_items} de {total_items_lyl}")



==================== ANÁLISIS: CUSTOMERS ====================
Dimensiones: (387, 26)

--- Valores Nulos ---
               Total Nulos    % Nulos
fit_top                  1   0.258398
fit_bottom               1   0.258398
size_footwear            6   1.550388
size_bra                23   5.943152
size_cup                33   8.527132
weight                   1   0.258398
adventurous              2   0.516796
prices                   2   0.516796
job                      2   0.516796
date_birth               3   0.775194
age                      3   0.775194
is_mother               20   5.167959
style_1                  1   0.258398
style_2                 49  12.661499

--- Tipos de Datos (Head) ---
user_id                   object
user_market               object
frequency                 object
newsletter_subscribed      int64
dress_leisure             object
dress_work                object
fit_top                   object
fit_bottom                object
body_shape                

In [56]:
# 1. Para TODAS las columnas de TEXTO (Categorías): Rellenar con "Desconocido"
cols_texto = customers.select_dtypes(include=['object', 'category']).columns

for col in cols_texto:
    # Si la columna es tipo 'category' (pandas), primero añadimos la categoría nueva
    if customers[col].dtype.name == 'category':
        customers[col] = customers[col].cat.add_categories('Desconocido')
    
    # Rellenamos los NAs
    customers[col] = customers[col].fillna('Desconocido')

# 2. Para TODAS las columnas NUMÉRICAS: Rellenar con la MEDIANA
cols_num = customers.select_dtypes(include=['number']).columns

# Calcula la mediana de cada columna y rellena los huecos
customers[cols_num] = customers[cols_num].fillna(customers[cols_num].median())

# Verificación rápida
print("NAs restantes:", customers.isna().sum().sum())

NAs restantes: 0


In [57]:
# 1. Para las columnas de TEXTO (Categorías): Rellena los huecos con "No_Aplica"
cols_texto = items.select_dtypes(include=['object', 'category']).columns
items[cols_texto] = items[cols_texto].fillna('No_Aplica')

# 2. Para las columnas NUMÉRICAS: Rellena los huecos con -1
# (Usamos -1 para que el modelo distinga entre valor 0 real y dato no existente)
cols_num = items.select_dtypes(include=['number']).columns
items[cols_num] = items[cols_num].fillna(-1)

In [58]:

lyl_customers = pd.merge(lyl, customers, on='user_id', how='left')

df_final = pd.merge(lyl_customers, items, on='product_variant_id', how='left')



In [59]:
cols_to_drop = ['product_variant_id', 'user_id']

df_final = df_final.drop(columns=cols_to_drop, errors='ignore')
#Se borran las filas restantes con NAs debido a que hay clientas que no tienen nuinguna prenda asociada
df_final = df_final.dropna()


df=df_final
df2=df_final


In [60]:
# Asegurar que response es booleana/binaria
df["response"] = df["response"].astype(str).str.strip().isin(["True", "1", "true"])

# Variable objetivo como 0/1
df["response_int"] = df["response"].astype(int)


# 2.PASO #

In [61]:
df

,response,place,occurred_on_,user_market,frequency,newsletter_subscribed,dress_leisure,dress_work,fit_top,fit_bottom,...,sleeve_long_cm,sole_length,style,thicknees,toecap,type_of_length,waist_contour,weather,current_price_eur,response_int
0,True,look-and-like,2025-01-26 11:17:28.849743,FR,on-demand,0,street,femenine,straight,straight,...,-1.0,No_Aplica,"casual,street",No_Aplica,No_Aplica,long,39.0,"cold,cold_season",10995.0,1
1,False,look-and-like,2025-01-19 09:44:08.914438,FR,bimonthly,0,street,street,loose,straight,...,-1.0,No_Aplica,"boho,street",No_Aplica,No_Aplica,No_Aplica,-1.0,"cold_season,warm_season",3790.0,0
2,True,look-and-like,2025-01-01 08:28:42.628929,FR,on-demand,0,classic,street,loose,loose,...,61.0,No_Aplica,"casual,classic",No_Aplica,No_Aplica,No_Aplica,47.0,"cold,cold_season",3999.0,1
3,False,checkout-welcome,2025-01-01 10:10:41.000233,FR,on-demand,0,street,femenine,fitted,fitted,...,64.0,No_Aplica,"boho,street",No_Aplica,No_Aplica,No_Aplica,54.0,cold,2999.0,0
4,False,look-and-like,2025-01-07 10:56:10.280325,FR,on-demand,0,femenine,classic,loose,straight,...,5.0,No_Aplica,"boho,classic",No_Aplica,No_Aplica,No_Aplica,64.0,warm_season,4190.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349257,True,look-and-like,2025-02-20 07:07:40.700267,DE,on-demand,0,femenine,femenine,loose,fitted,...,-1.0,No_Aplica,"classic,minimal",No_Aplica,round,No_Aplica,-1.0,"warm,warm_season",10500.0,1
349258,False,look-and-like,2025-02-22 23:10:36.037319,UK,on-demand,0,street,femenine,straight,straight,...,11.0,No_Aplica,"casual,street",No_Aplica,No_Aplica,knee,40.0,"warm,warm_season",5999.0,0
349259,True,look-and-like,2025-02-03 05:49:06.637680,FR,bimonthly,0,classic,street,straight,straight,...,66.0,No_Aplica,"classic,minimal",No_Aplica,No_Aplica,No_Aplica,49.0,"cold_season,warm_season",3999.0,1
349260,False,look-and-like,2025-02-01 07:01:04.701791,ES,on-demand,1,casual,classic,fitted,straight,...,-1.0,3_0,"casual,street",No_Aplica,round,No_Aplica,-1.0,"cold_season,warm_season",4990.0,0


In [62]:
print(df.dtypes)

# Ver columnas que tienen mezcla de bool y str
for col in df.columns:
    tipos = set(type(v) for v in df[col].dropna())
    if len(tipos) > 1:
        print(col, tipos)


response                bool
place                 object
occurred_on_          object
user_market           object
frequency             object
                      ...   
type_of_length        object
waist_contour        float64
weather               object
current_price_eur    float64
response_int           int64
Length: 74, dtype: object
is_mother {<class 'str'>, <class 'bool'>}
back_neckline {<class 'str'>, <class 'bool'>}
basic {<class 'str'>, <class 'bool'>}
chest_volume {<class 'str'>, <class 'bool'>}
cover {<class 'str'>, <class 'bool'>}
elasticated_lining {<class 'str'>, <class 'bool'>}
hips_volume {<class 'str'>, <class 'bool'>}
light {<class 'str'>, <class 'bool'>}
rubber_waist {<class 'str'>, <class 'bool'>}
shoulders_pad {<class 'str'>, <class 'bool'>}


In [63]:
for col in df.columns:
    if df[col].dtype == 'object':
        # Convertimos todo a string para uniformizar
        df[col] = df[col].astype(str)

In [64]:
print(df.dtypes)

# Ver columnas que tienen mezcla de bool y str
for col in df.columns:
    tipos = set(type(v) for v in df[col].dropna())
    if len(tipos) > 1:
        print(col, tipos)


response                bool
place                 object
occurred_on_          object
user_market           object
frequency             object
                      ...   
type_of_length        object
waist_contour        float64
weather               object
current_price_eur    float64
response_int           int64
Length: 74, dtype: object


In [65]:
cols_to_drop = ['occurred_on_', 'model','composition_detail','']

df = df.drop(columns=cols_to_drop, errors='ignore')

In [66]:
# import pandas as pd
# from sklearn.utils import resample


# df_false = df[df.response == False]
# df_true = df[df.response == True]

# df_false_downsampled = resample(df_false, 
#                                 replace=False,    
#                                 n_samples=len(df_true), 
#                                 random_state=42) 


# df = pd.concat([df_false_downsampled, df_true])

# print("Undersampling resultado:")
# print(df.response.value_counts())

In [67]:
df_modelo=df

In [68]:
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Importar los competidores
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
# E. Definir Target y Features
target_col = 'response_int' if 'response_int' in df.columns else 'response'
# Ajuste por si el target era texto y se codificó
if target_col not in df_modelo.columns:
    target_col = [c for c in df_modelo.columns if c.startswith('response_')][0]

X = df_modelo.drop(columns=[c for c in df_modelo.columns if c.startswith('response')])
y = df_modelo[target_col]

# F. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=30)

In [69]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Modelos
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# -----------------------------------------------------------------------------
# 1. CARGA Y LIMPIEZA (Tu código robusto)
# -----------------------------------------------------------------------------
# Cargar datos (Asegúrate de tener el df cargado o leerlo aquí)
# df = pd.read_csv('dfchati.csv') # Descomenta si necesitas leerlo de nuevo

# (Asumiendo que df ya está en memoria y limpio de tu paso anterior. 
# Si no, ejecuta las limpiezas de tipos mixtos y NAs aquí mismo).
# Repito la limpieza rápida por seguridad:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str) # Arreglar mixtos

cols_texto = df.select_dtypes(include=['object']).columns
df[cols_texto] = df[cols_texto].fillna('No_Aplica')
cols_num = df.select_dtypes(include=['number']).columns
df[cols_num] = df[cols_num].fillna(-1)

# Borrar columnas ID/Fecha masivas para evitar crash
limite_cardinalidad = 50
cols_borrar = [c for c in df.columns if df[c].dtype == 'object' and df[c].nunique() > limite_cardinalidad]
df_limpio = df.drop(columns=cols_borrar)

# Encoding
print("🔄 Generando Dummies...")
df_modelo = pd.get_dummies(df_limpio, drop_first=True)

# -----------------------------------------------------------------------------
# 2. PREPARAR DATOS
# -----------------------------------------------------------------------------
target_col = 'response_int' if 'response_int' in df.columns else 'response'
if target_col not in df_modelo.columns:
    target_col = [c for c in df_modelo.columns if c.startswith('response_')][0]

X = df_modelo.drop(columns=[c for c in df_modelo.columns if c.startswith('response')])
y = df_modelo[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -----------------------------------------------------------------------------
# 3. SELECCIÓN DE CARACTERÍSTICAS (Crucial para velocidad)
# -----------------------------------------------------------------------------
print(f"📉 Reduciendo dimensiones (Original: {X_train.shape[1]} columnas)...")
k_best = 300
selector = SelectKBest(score_func=f_classif, k=k_best)

X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)
print("✅ Datos listos para el torneo.")

# -----------------------------------------------------------------------------
# 4. DEFINICIÓN DE COMPETIDORES
# -----------------------------------------------------------------------------
modelos = {
    # MODELO 1: Regresión Logística (El matemático)
    # class_weight='balanced' fuerza a prestar atención a la clase minoritaria
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear'),
    
    # MODELO 2: Random Forest (El clásico)
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', n_jobs=-1, random_state=42),
    
    # MODELO 3: Extra Trees (El aleatorio - suele reducir overfitting)
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, max_depth=15, class_weight='balanced', n_jobs=-1, random_state=42),
    
    # MODELO 4: HistGradientBoosting (El rápido y moderno - tipo LightGBM)
    # Este suele ser el MEJOR en precisión pura, aunque no tiene class_weight nativo fácil en versiones antiguas
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=100, max_depth=10, random_state=42),
    
    # MODELO 5: AdaBoost (El especialista en corrección de errores)
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    
    # MODELO 6: Decision Tree Simple (La referencia rápida)
    "Decision Tree": DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42)
}

# -----------------------------------------------------------------------------
# 5. EL TORNEO
# -----------------------------------------------------------------------------
resultados = []

print("\n🏆 INICIANDO TORNEO DE MODELOS...")
print(f"{'MODELO':<25} | {'ACCURACY':<10} | {'F1 (GLOBAL)':<12} | {'F1 (COMPRAS)':<12} | {'TIEMPO'}")
print("-" * 85)

for nombre, modelo in modelos.items():
    start = time.time()
    
    try:
        modelo.fit(X_train_sel, y_train)
        y_pred = modelo.predict(X_test_sel)
        
        # Métricas
        acc = accuracy_score(y_test, y_pred)
        f1_w = f1_score(y_test, y_pred, average='weighted')
        f1_pos = f1_score(y_test, y_pred, pos_label=1) # El que te importa
        
        tiempo = time.time() - start
        
        print(f"{nombre:<25} | {acc:.4f}     | {f1_w:.4f}       | {f1_pos:.4f}       | {tiempo:.2f}s")
        
        resultados.append({
            'Modelo': nombre,
            'Accuracy': acc,
            'F1_Global': f1_w,
            'F1_Compras': f1_pos, # Usaremos este para ordenar el ganador
            'Tiempo': tiempo
        })
    except Exception as e:
        print(f"{nombre:<25} | ❌ ERROR: {str(e)[:50]}...")

# -----------------------------------------------------------------------------
# 6. GANADOR Y CONCLUSIONES
# -----------------------------------------------------------------------------
df_res = pd.DataFrame(resultados).sort_values(by='F1_Compras', ascending=False)

print("\n" + "="*40)
print("       CLASIFICACIÓN FINAL (Por F1 Compras)")
print("="*40)
print(df_res[['Modelo', 'F1_Compras', 'Accuracy', 'F1_Global']])

ganador = df_res.iloc[0]
print(f"\n🥇 EL MEJOR MODELO ES: {ganador['Modelo']}")
print(f"   -> Capaz de detectar compras con un score de: {ganador['F1_Compras']:.4f}")

# Imprimir reporte detallado del ganador
print(f"\n--- Reporte del Ganador ({ganador['Modelo']}) ---")
# Re-entrenamos o recuperamos el ganador (en este script simple, ya está entrenado en el bucle, 
# pero para mostrar el reporte hay que usar el último del bucle o guardarlo. 
# Aquí mostramos el reporte genérico de cómo interpretarlo).
print("Revisa la tabla de arriba. Si el F1_Compras sigue bajo (<0.5),")
print("necesitamos Ingeniería de Variables (crear nuevas columnas) más que cambiar de modelo.")

🔄 Generando Dummies...
📉 Reduciendo dimensiones (Original: 443 columnas)...
✅ Datos listos para el torneo.

🏆 INICIANDO TORNEO DE MODELOS...
MODELO                    | ACCURACY   | F1 (GLOBAL)  | F1 (COMPRAS) | TIEMPO
-------------------------------------------------------------------------------------
Logistic Regression       | 0.6138     | 0.6209       | 0.5356       | 151.21s
Random Forest             | 0.7147     | 0.7167       | 0.6213       | 53.80s
Extra Trees               | 0.7113     | 0.7143       | 0.6241       | 63.74s
HistGradientBoosting      | 0.7123     | 0.6848       | 0.4807       | 69.67s
AdaBoost                  | 0.6547     | 0.5777       | 0.2268       | 399.45s
Decision Tree             | 0.6476     | 0.6523       | 0.5510       | 10.57s

       CLASIFICACIÓN FINAL (Por F1 Compras)
                 Modelo  F1_Compras  Accuracy  F1_Global
2           Extra Trees    0.624131  0.711306   0.714285
1         Random Forest    0.621319  0.714670   0.716731
5        

In [70]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score, accuracy_score

# 1. CARGA Y LIMPIEZA INICIAL
# -----------------------------------------------------------------------------

# Limpieza general de tipos (evita errores de texto/número)
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str)

# Imputación de NAs en variables predictoras
cols_texto = df.select_dtypes(include=['object']).columns
df[cols_texto] = df[cols_texto].fillna('No_Aplica')
cols_num = df.select_dtypes(include=['number']).columns
df[cols_num] = df[cols_num].fillna(-1)

# LIMPIEZA CRÍTICA DEL TARGET (Soluciona el bajo F1 anterior)
# -----------------------------------------------------------------------------
# Identificamos el target
target_col = 'response_int' if 'response_int' in df.columns else 'response'

# Si el target tiene -1 (nulos imputados), BORRAMOS esas filas. 
# No podemos aprender de datos sin respuesta.
if target_col in df.columns:
    n_borrados = len(df[df[target_col] == -1])
    if n_borrados > 0:
        print(f"⚠️ Eliminando {n_borrados} filas sin respuesta válida (target = -1)...")
        df = df[df[target_col] != -1]

# Borrar columnas ID que confunden memoria
limite_cardinalidad = 50
cols_borrar = [c for c in df.columns if df[c].dtype == 'object' and df[c].nunique() > limite_cardinalidad]
df = df.drop(columns=cols_borrar)

# 2. FUNCIÓN MAESTRA DE ENTRENAMIENTO
# -----------------------------------------------------------------------------
def entrenar_experto(df_subset, nombre="Experto"):
    # Encoding local (solo crea columnas para lo que existe en este subgrupo)
    df_encoded = pd.get_dummies(df_subset, drop_first=True)
    
    # Re-localizar target tras encoding
    tgt = target_col
    if tgt not in df_encoded.columns:
        # Si era texto, buscar la dummy generada (ej: response_True)
        matches = [c for c in df_encoded.columns if c.startswith('response_')]
        if not matches: return None # Error
        tgt = matches[0]

    X = df_encoded.drop(columns=[c for c in df_encoded.columns if c.startswith('response')])
    y = df_encoded[tgt]
    
    # Filtro de seguridad: Mínimo 50 datos y 2 clases para entrenar
    if len(X) < 50 or y.nunique() < 2:
        return None

    # Entrenar
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = ExtraTreesClassifier(n_estimators=100, max_depth=15, class_weight='balanced', n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    return {
        'Segmento': nombre,
        'F1_Compras': f1_score(y_test, y_pred, pos_label=1), # Asumiendo 1 = Compra/True
        'Accuracy': accuracy_score(y_test, y_pred),
        'N_Datos': len(y_test)
    }

# 3. COMPARATIVA: GLOBAL VS SEGMENTADO
# -----------------------------------------------------------------------------
print("\n--- 1. Entrenando Modelo GLOBAL (Línea Base) ---")
res_global = entrenar_experto(df, "GLOBAL")
print(f"📊 F1 Global Actual: {res_global['F1_Compras']:.4f}")

print("\n--- 2. Entrenando Modelos EXPERTOS (Por Familia) ---")
col_familia = 'family' # Asegúrate de que esta columna existe
resultados = []

# Iteramos por cada familia de prenda
if col_familia in df.columns:
    for familia in df[col_familia].unique():
        # Filtramos datos de esa familia
        df_fam = df[df[col_familia] == familia].copy()
        # Borramos la columna familia (ya es redundante aquí)
        df_fam = df_fam.drop(columns=[col_familia])
        
        print(f"   ⚙️ Entrenando experto para: {familia}...", end=" ")
        res = entrenar_experto(df_fam, str(familia))
        
        if res:
            print(f"✅ F1: {res['F1_Compras']:.4f}")
            resultados.append(res)
        else:
            print("❌ (Insuficientes datos)")
else:
    print(f"ERROR: No encuentro la columna '{col_familia}' para segmentar.")

# 4. INFORME FINAL
# -----------------------------------------------------------------------------
if resultados:
    df_res = pd.DataFrame(resultados)
    
    # Calculamos la media ponderada del F1 Segmentado (para ser justos con el Global)
    f1_ponderado = np.average(df_res['F1_Compras'], weights=df_res['N_Datos'])
    
    print("\n" + "="*50)
    print("           VEREDICTO FINAL")
    print("="*50)
    print(f"🌍 Modelo Único (Global):         {res_global['F1_Compras']:.4f}")
    print(f"🧩 Modelos Segmentados (Media):   {f1_ponderado:.4f}")
    
    mejora = (f1_ponderado - res_global['F1_Compras']) * 100
    if mejora > 0:
        print(f"🚀 CONCLUSIÓN: Segmentar MEJORA el resultado un +{mejora:.2f}%")
    else:
        print(f"📉 CONCLUSIÓN: Segmentar NO mejora (El modelo global ya aprendía bien).")
        
    print("\nDetalle por Experto:")
    print(df_res[['Segmento', 'F1_Compras', 'N_Datos']].sort_values(by='F1_Compras', ascending=False))


--- 1. Entrenando Modelo GLOBAL (Línea Base) ---


KeyboardInterrupt: 